# 03 3DEP Offset Diagnosis

Purpose:
- read outputs from `diagnose_3dep_offsets.py`,
- inspect empirical offsets, formal transformation residuals, and pair consistency,
- keep the notebook lightweight instead of re-running heavy matching.

Scientific notes:
- CASALS refh is treated here as WGS84 ellipsoidal height.
- 3DEP products may use NAVD88 and project-specific frames.
- Pair-to-pair interpretation must keep product semantics and vertical datum issues explicit.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = Path('../diagnose_3dep_offsets')
SUMMARY_CSV = OUT_DIR / 'per_pair_summary.csv'
SUMMARY_JSON = OUT_DIR / 'per_pair_summary.json'
summary = pd.read_csv(SUMMARY_CSV)
payload = json.loads(SUMMARY_JSON.read_text(encoding='utf-8'))
summary

In [ ]:
cols = [c for c in summary.columns if 'offset' in c or 'residual' in c]
print('offset-related columns:', cols)
summary[['label'] + cols[:8]] if cols else summary[['label']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary.plot.bar(x='label', y='empirical_offset_m', ax=axes[0], color='tab:blue', legend=False)
axes[0].set_title('Empirical offset by pair')
axes[0].set_ylabel('meters')
formal_cols = [c for c in summary.columns if c.startswith('formal_') and c.endswith('_median_m')]
if formal_cols:
    summary.plot(x='label', y=formal_cols, marker='o', ax=axes[1])
    axes[1].set_title('Formal transformation medians')
    axes[1].set_ylabel('meters')
else:
    axes[1].text(0.5, 0.5, 'No formal columns found', ha='center', va='center')
    axes[1].set_axis_off()
fig.tight_layout()

In [ ]:
for label, report in payload.get('pair_reports', {}).items():
    print('\nPAIR:', label)
    print('scientific_notes:', report.get('scientific_notes', []))
    print('matching:', report.get('matching', {}))
    print('empirical_offset:', report.get('empirical_offset', {}))